In [4]:
!pip install monai torch torchvision pillow numpy einops


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import torch
import argparse
from monai.networks.nets.swin_unetr import SwinTransformer

In [6]:
feature_size = 128
in_channels = 3
pretrained_checkpoint = "echocare_encoder.pth"   
use_checkpoint = True          

encoder = SwinTransformer(
    in_chans=in_channels,
    embed_dim=feature_size,
    window_size=[8] * 2,
    patch_size=[2] * 2,
    depths=[2, 2, 18, 2],
    num_heads=[4, 8, 16, 32],
    mlp_ratio=4.0,
    qkv_bias=True,
    use_checkpoint=use_checkpoint,
    spatial_dims=2,
    use_v2=True)

if pretrained_checkpoint is not None:
    model_dict = torch.load(pretrained_checkpoint, map_location=torch.device('cpu'))
    state_dict = model_dict
    state_dict.pop('mask_token')
    encoder.load_state_dict(state_dict, strict=True)
    print("Using pretrained self-supervised Swin Transformer backbone weights !")

Using pretrained self-supervised Swin Transformer backbone weights !


In [7]:
# Test case: forward pass with dummy input
# Expected output feature map shapes:
# [1, 128, 128, 128], [1, 256, 64, 64], [1, 512, 32, 32], [1, 1024, 16, 16], [1, 2048, 8, 8]
x = torch.rand(1, 3, 256, 256)
x_outs = encoder(x)
print([x_out.shape for x_out in x_outs])

[torch.Size([1, 128, 128, 128]), torch.Size([1, 256, 64, 64]), torch.Size([1, 512, 32, 32]), torch.Size([1, 1024, 16, 16]), torch.Size([1, 2048, 8, 8])]
